# SBI2 data staging and first-pass QC

This notebook previews and optionally downloads the processed Altair, POSH, and Aconcagua profiles, then checks file integrity, schemas, and guide overlap. The Altair phase selection includes both `phase_guide.h5ad` (one profile per guide) and `phase_guide_x_screen.h5ad` (one profile per guide per screen). The Altair marker archive is also downloaded for the 4i guide profiles. The separately supplied Aconcagua guide/gene Parquet bundle is checksum-verified and copied from a local folder; it has no remote URL configured here. This notebook does not fetch images or Aconcagua cell-level data.

Before enabling downloads, review `docs/DATASETS.md` and make sure the selected data volume has enough free space.

In [ ]:
from pathlib import Path
import os, subprocess, sys

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / 'scripts' / 'download_data.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Start Jupyter from this repository or open this notebook from the repository.')
DATA_DIR = Path(os.environ.get('SBI2_DATA_DIR', ROOT / 'data')).expanduser().resolve()
ACON_GUIDE_SOURCE = Path(os.environ.get('ACONCAGUA_GUIDE_SOURCE_DIR', ROOT.parent / 'aconcagua_guide_gene_features')).expanduser().resolve()
STAGE_ACONCAGUA_GUIDE_DATA = False
PRESET = os.environ.get('DATA_PRESET', 'core')
RUN_DOWNLOAD = False  # Change to True only after reviewing the size preview.
print('Repository:', ROOT)
print('Data directory:', DATA_DIR)
print('Preset:', PRESET)
print('Aconcagua guide/gene source:', ACON_GUIDE_SOURCE)
print('Source folder found:', ACON_GUIDE_SOURCE.is_dir())

## 1. Preview source files and sizes

The preview checks current remote response headers and does not write dataset files. The default core selection now includes `phase_guide_x_screen.h5ad` (3,132,434,691 bytes, about 2.92 GiB), in addition to the 1.90 GiB marker archive. Plan for approximately 5.24 GB of network transfer before extraction and other files.

In [ ]:
preview = subprocess.run([sys.executable, str(ROOT / 'scripts' / 'download_data.py'), '--preset', PRESET, '--data-dir', str(DATA_DIR), '--dry-run'], cwd=ROOT, text=True, capture_output=True)
print(preview.stdout)
if preview.stderr:
    print(preview.stderr)
if preview.returncode:
    raise RuntimeError(f'Dry-run failed with exit code {preview.returncode}')

## 2. Download selected datasets

Downloads are disabled by default. Set `RUN_DOWNLOAD = True` above after reviewing every listed object and its size. The script will ask for confirmation again.

In [ ]:
if RUN_DOWNLOAD:
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'download_data.py'), '--preset', PRESET, '--data-dir', str(DATA_DIR), '--yes'], cwd=ROOT, check=True)
else:
    print('Download skipped. Set RUN_DOWNLOAD = True after reviewing the preview.')

## 3. Stage the supplied Aconcagua guide/gene-level profiles

This six-file Parquet bundle is supplied in `aconcagua_guide_gene_features/` alongside the prep repository. It contains guide and gene summaries for all-cell, interphase, and mitotic populations. It is not part of the HTTP downloader because no public file URL was provided. Set `ACONCAGUA_GUIDE_SOURCE_DIR` if the folder is elsewhere. The dry run verifies all MD5 values and previews the roughly 624 MiB copy.

In [ ]:
stage_script = ROOT / 'scripts' / 'stage_aconcagua_guide_data.py'
if ACON_GUIDE_SOURCE.is_dir():
    stage_preview = subprocess.run([sys.executable, str(stage_script), '--source-dir', str(ACON_GUIDE_SOURCE), '--data-dir', str(DATA_DIR), '--dry-run'], cwd=ROOT, text=True, capture_output=True)
    print(stage_preview.stdout)
    if stage_preview.stderr:
        print(stage_preview.stderr)
    if stage_preview.returncode:
        raise RuntimeError(f'Aconcagua guide-data preflight failed with exit code {stage_preview.returncode}')
else:
    print('Aconcagua guide bundle not found. Supply it to the VM and set ACONCAGUA_GUIDE_SOURCE_DIR.')

Set `STAGE_ACONCAGUA_GUIDE_DATA = True` in the setup cell to copy the checksum-verified files into `data/aconcagua_guide_gene_features/`. The toggle is the confirmation for copying the files.

In [ ]:
if STAGE_ACONCAGUA_GUIDE_DATA:
    if not ACON_GUIDE_SOURCE.is_dir():
        raise FileNotFoundError(f'Aconcagua guide-data source folder not found: {ACON_GUIDE_SOURCE}')
    subprocess.run([sys.executable, str(stage_script), '--source-dir', str(ACON_GUIDE_SOURCE), '--data-dir', str(DATA_DIR), '--yes'], cwd=ROOT, check=True)
else:
    print('Staging skipped. Set STAGE_ACONCAGUA_GUIDE_DATA = True after reviewing the verified file list.')

## 4. Run first-pass quality checks

The report is saved under `<data-dir>/qc/quality_report.json`. It checks file hashes, H5AD/Parquet structure, metadata fields, and guide/target overlap of OPS0077 profiles against both Altair phase summaries. The Aconcagua Parquet checks include guide barcode uniqueness, target consistency, cell-count summaries, schema, and shared guide IDs across populations. Cross-atlas comparisons use target gene symbols; Aconcagua `cell_barcode_0` values are not assumed to be interchangeable with Altair guide sequences.

In [ ]:
qc_cmd = [sys.executable, str(ROOT / 'scripts' / 'run_quality_checks.py'), '--data-dir', str(DATA_DIR), '--outdir', str(DATA_DIR / 'qc'), '--allow-incomplete']
qc = subprocess.run(qc_cmd, cwd=ROOT, text=True, capture_output=True)
print(qc.stdout)
if qc.stderr:
    print(qc.stderr)
if qc.returncode:
    raise RuntimeError(f'Quality checks reported an error; inspect {DATA_DIR / "qc" / "quality_report.json"}')
